# Infer missing subDistrict

This notebook finds charges where `place_of_offence.address` is present but `place_of_offence.subDistrict` is missing.

For each such charge, it uses GPT-5.4 (Azure OpenAI) to infer the subDistrict from the address string, writes the inferred subDistrict back to `verified-features`, and exports an Excel file of cases where subDistrict could not be identified.

The default query filters out `exclude=True` records to match the active verified dataset used elsewhere in the repo.

In [3]:
import json
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Optional
import re

import pandas as pd
from bson import ObjectId
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from pymongo import MongoClient, UpdateOne

# Add parent to path so we can import featureExtraction modules
sys.path.insert(0, str(Path.cwd().parent))
from featureExtraction.utils.hkDistricts import SubDistrict, get_district_for_subdistrict

# Load env (try multiple locations like infer-missing-ages.ipynb)
for dotenv_path in (
    Path.cwd() / '.env',
    Path.cwd() / 'notebooks' / '.env',
    Path.cwd().parent / 'notebooks' / '.env',
    Path.cwd().parent / 'featureExtraction' / '.env',
):
    if dotenv_path.exists():
        load_dotenv(dotenv_path)
        break

# --- MongoDB connection ---
uri = os.getenv('DB_MONGODB_URI')
if not uri:
    raise RuntimeError('DB_MONGODB_URI is not set')

client = MongoClient(uri)
db = client.get_database(os.getenv('DB_NAME', 'drug-sentencing-predictor'))
verified_features = db.get_collection('verified-features')

# --- OpenAI client (Azure, gpt-5.4) ---
api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_BASE_URL')
if not api_key or not base_url:
    raise RuntimeError('OPENAI_API_KEY and OPENAI_BASE_URL must be set')

openai_client = OpenAI(api_key=api_key, base_url=base_url)
MODEL = os.getenv('MODEL', 'gpt-5.4')

pd.set_option('display.max_colwidth', None)

active_only = True


In [4]:
# Build the set of valid sub-district strings (from the enum)
VALID_SUBDISTRICTS = {sub.value for sub in SubDistrict}

print(f'Loaded {len(VALID_SUBDISTRICTS)} valid sub-districts from hkDistricts enum')
print(f'Sample: {sorted(VALID_SUBDISTRICTS)[:10]}...')

# Pydantic model for structured GPT response
class SubDistrictInference(BaseModel):
    subDistrict: Optional[str] = Field(
        description="The inferred sub-district name exactly as listed in the Hong Kong sub-district list. "
        "Set to null if the address does not clearly fall into one of the known sub-districts."
    )
    reason: str = Field(
        description="Brief reason for the inference (e.g., which landmarks/clues were used) "
        "or explanation for why it could not be determined."
    )


Loaded 127 valid sub-districts from hkDistricts enum
Sample: ['Aberdeen', 'Admiralty', 'Ap Lei Chau', 'Beacon Hill', 'Braemar Hill', 'Causeway Bay', 'Central', 'Chai Wan', 'Cheung Chau', 'Cheung Muk Tau']...


In [5]:
# Build the list of valid sub-district strings for the prompt
subdistrict_list_str = "\n".join(sorted(f"  - {s}" for s in VALID_SUBDISTRICTS))


def infer_subdistrict(address: str) -> SubDistrictInference:
    """Call GPT-5.4 to infer the HK sub-district from an address string.
    Returns a SubDistrictInference with the subDistrict value or None."""
    prompt = (
        "You are a Hong Kong geography specialist. Given an address in Hong Kong, determine "
        "which of the following sub-districts it belongs to.\n\n"
        "Valid sub-districts:\n"
        f"{subdistrict_list_str}\n\n"
        "Rules:\n"
        "1. Return the EXACT sub-district name as listed above (case-sensitive).\n"
        "2. If the address clearly matches a sub-district, return it.\n"
        "3. If the address is too vague, ambiguous, or does not match any known sub-district, "
        "set subDistrict to null and explain why.\n"
        "4. For well-known areas (e.g., 'Tsing Yi' → 'Tsing Yi', 'Mong Kok' → 'Mong Kok'), "
        "return the matching sub-district.\n"
        "5. For estate names, infer the approximate sub-district (e.g., 'Cheung Hong Estate' is in Tsing Yi).\n"
        "6. Do NOT guess if you are unsure — it is better to return null than a wrong sub-district."
    )

    response = openai_client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": f"Address: {address}"},
        ],
        text_format=SubDistrictInference,
    )

    result: SubDistrictInference = response.output_parsed

    # Validate that the returned subDistrict is one of the valid enum values
    if result.subDistrict is not None and result.subDistrict not in VALID_SUBDISTRICTS:
        print(f"  WARNING: GPT returned invalid subDistrict '{result.subDistrict}' for address '{address}'")
        return SubDistrictInference(subDistrict=None, reason=f"Invalid sub-district returned: {result.subDistrict}")

    return result


# Test with a known address
test_result = infer_subdistrict("Room 1140, Hong Tai House, Cheung Hong Estate, Tsing Yi, New Territories")
print(f"Test address → subDistrict: {test_result.subDistrict}, reason: {test_result.reason}")


Test address → subDistrict: Tsing Yi, reason: Cheung Hong Estate is a public housing estate located on Tsing Yi Island, and the address explicitly includes 'Tsing Yi'.


In [7]:
# --- Query: find charges with address but no subDistrict ---

pipeline = []
if active_only:
    pipeline.append({'$match': {'exclude': {'$ne': True}}})

pipeline.extend([
    # Unwind charges to work at the charge level
    {'$unwind': '$judgement.charges'},
    # Filter to charges where place_of_offence exists and has a meaningful address
    {'$match': {
        'judgement.charges.place_of_offence': {'$ne': None},
        '$expr': {
            '$and': [
                # address must be non-null and non-empty
                {'$ne': ['$judgement.charges.place_of_offence.address', None]},
                {'$ne': ['$judgement.charges.place_of_offence.address', '']},
                # subDistrict must be null or empty string
                {'$or': [
                    {'$eq': ['$judgement.charges.place_of_offence.subDistrict', None]},
                    {'$eq': ['$judgement.charges.place_of_offence.subDistrict', '']},
                ]},
            ]
        },
    }},
    # Project the fields we need
    {
        '$project': {
            '_id': 1,
            'neutral_citation': '$judgement.neutral_citation',
            'charge_no': '$judgement.charges.charge_no',
            'address': '$judgement.charges.place_of_offence.address',
            'nature': '$judgement.charges.place_of_offence.nature',
            'subDistrict': '$judgement.charges.place_of_offence.subDistrict',
        }
    },
])

rows = list(verified_features.aggregate(pipeline, allowDiskUse=True))
print(f'Found {len(rows)} charges with address but missing subDistrict')

if rows:
    df = pd.DataFrame(rows)
    # Normalize ObjectId
    df['_id'] = df['_id'].apply(str)
    display(df.head(20))
else:
    print('No missing subDistrict cases found — all charges have subDistrict populated!')


Found 417 charges with address but missing subDistrict


,_id,neutral_citation,charge_no,address,nature,subDistrict
0,69c918e441df75d9210286be,[2021] HKDC 747,1,被告人所居住的單位,Residential building,None
1,69cf2b8dc96004cf50f0c5d3,[2025] HKCFI 1364,1,in the street near his home,Street,None
2,69cf2b8dc96004cf50f0c5d3,[2025] HKCFI 1364,2,inside the defendant's bedroom at his home,Residential building,None
3,69cf48d213addb9916257083,[2021] HKCFI 574,2,D1's residence,Residential building,None
4,69cf48de13addb9916257084,[2022] HKCFI 2407,2,Building - Unknown,Other,None
5,69cf4951a9f43af234c719ca,[2021] HKCFI 98,1,Post Office,Government or public building,None
6,69cf734bf9fafa2a2fe7d288,[2025] HKCFI 4134,1,Flat (6th floor of a building),Residential building,None
7,69cf734bf9fafa2a2fe7d288,[2025] HKCFI 4134,2,Flat (the said flat rented by the defendant),Residential building,None
8,69d0c87aa8276670f96f1761,[2023] HKDC 1856,1,Not Given,Other,None
9,69d373e2626c401b8b1c72da,[2023] HKCFI 925,1,Taxi,Public transport,None


In [8]:
# --- Process each charge with GPT to infer subDistrict ---

MAX_WORKERS = 5  # Adjust based on API rate limits

results = []


def process_row(row):
    """Process a single row: infer subDistrict from address."""
    address = row['address']
    inference = infer_subdistrict(address)
    return {
        '_id': str(row['_id']),
        'charge_no': row['charge_no'],
        'neutral_citation': row['neutral_citation'],
        'address': address,
        'inferred_subDistrict': inference.subDistrict,
        'inferred_district': (
            get_district_for_subdistrict(SubDistrict(inference.subDistrict)).value
            if inference.subDistrict is not None else None
        ),
        'reason': inference.reason,
    }


if rows:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_row, row): i for i, row in enumerate(rows)}
        for future in as_completed(futures):
            idx = futures[future]
            try:
                result = future.result()
                results.append(result)
                status = '✅' if result['inferred_subDistrict'] else '❌'
                print(f"[{idx + 1}/{len(rows)}] {status} {result['neutral_citation']} "
                      f"Ch.{result['charge_no']}: {result['inferred_subDistrict'] or 'UNRESOLVED'} "
                      f"({result['address'][:60]})")
            except Exception as exc:
                print(f"[{idx + 1}/{len(rows)}] ❌ ERROR: {exc}")
                results.append({
                    '_id': str(rows[idx]['_id']),
                    'charge_no': rows[idx]['charge_no'],
                    'neutral_citation': rows[idx]['neutral_citation'],
                    'address': rows[idx]['address'],
                    'inferred_subDistrict': None,
                    'inferred_district': None,
                    'reason': f'API error: {exc}',
                })

    results_df = pd.DataFrame(results)
else:
    results_df = pd.DataFrame(columns=[
        '_id', 'charge_no', 'neutral_citation', 'address',
        'inferred_subDistrict', 'inferred_district', 'reason'
    ])

print(f"\nProcessed {len(results_df)} charges total")
print(f"  ✓ Inferred: {results_df['inferred_subDistrict'].notna().sum()}")
print(f"  ✗ Unresolved: {results_df['inferred_subDistrict'].isna().sum()}")


[1/417] ❌ [2021] HKDC 747 Ch.1: UNRESOLVED (被告人所居住的單位)
[4/417] ❌ [2021] HKCFI 574 Ch.2: UNRESOLVED (D1's residence)
[6/417] ❌ [2021] HKCFI 98 Ch.1: UNRESOLVED (Post Office)
[5/417] ❌ [2022] HKCFI 2407 Ch.2: UNRESOLVED (Building - Unknown)
[2/417] ❌ [2025] HKCFI 1364 Ch.1: UNRESOLVED (in the street near his home)
[7/417] ❌ [2025] HKCFI 4134 Ch.1: UNRESOLVED (Flat (6th floor of a building))
[3/417] ❌ [2025] HKCFI 1364 Ch.2: UNRESOLVED (inside the defendant's bedroom at his home)
[9/417] ❌ [2023] HKDC 1856 Ch.1: UNRESOLVED (Not Given)
[11/417] ❌ [2023] HKDC 784 Ch.5: UNRESOLVED (XD 2420 Car)
[8/417] ❌ [2025] HKCFI 4134 Ch.2: UNRESOLVED (Flat (the said flat rented by the defendant))
[10/417] ❌ [2023] HKCFI 925 Ch.1: UNRESOLVED (Taxi)
[12/417] ❌ [2021] HKCFI 1919 Ch.2: UNRESOLVED (the defendant's flat (residence))
[13/417] ✅ [2024] HKDC 226 Ch.1: Jordan Valley (九龍彩盈邨盈富樓7樓713室)
[14/417] ❌ [2024] HKDC 1370 Ch.1: UNRESOLVED (羅湖管制站海關入境大堂)
[18/417] ❌ [2025] HKDC 525 Ch.1: UNRESOLVED (涉案單位（單位內客廳）

In [9]:
# --- Write inferred subDistricts back to MongoDB ---

# Only write rows where we successfully inferred a subDistrict
inferred_rows = results_df[results_df['inferred_subDistrict'].notna()].to_dict('records')

update_operations = []
for row in inferred_rows:
    subdistrict_value = row['inferred_subDistrict']
    district_value = row['inferred_district']
    source_text = (
        f"Inferred from address '{row['address']}' using GPT-5.4: {row['reason']}"
    )

    update_operations.append(
        UpdateOne(
            {
                '_id': ObjectId(row['_id']),
                'judgement.charges.charge_no': row['charge_no'],
            },
            {
                '$set': {
                    'judgement.charges.$[charge].place_of_offence.subDistrict': subdistrict_value,
                    'judgement.charges.$[charge].place_of_offence.district': district_value,
                }
            },
            array_filters=[
                {'charge.charge_no': row['charge_no']},
            ],
        )
    )

update_result = None
if update_operations:
    update_result = verified_features.bulk_write(update_operations, ordered=False)
    print(f"Written {update_result.modified_count} charge(s) with inferred subDistricts")
else:
    print("No subDistricts to write back")


Written 291 charge(s) with inferred subDistricts


In [14]:
# --- Export unresolved cases to Excel ---
unresolved = results_df[results_df['inferred_subDistrict'].isna()].copy()

unresolved_columns = [
    'neutral_citation',
    'charge_no',
    'address',
    'reason',
]


def _sanitize_for_excel(value):
    """Remove Excel-illegal control characters from string values."""
    if isinstance(value, str):
        return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', value)
    return value


if not unresolved.empty:
    unresolved_out = unresolved[unresolved_columns].sort_values(
        ['neutral_citation', 'charge_no']
    ).reset_index(drop=True)
    # Sanitize all string columns to remove Excel-illegal characters
    unresolved_out = unresolved_out.map(_sanitize_for_excel)
    unresolved_out.to_excel('unresolved_subdistricts.xlsx', index=False)
    print(f"Exported {len(unresolved_out)} unresolved cases to 'unresolved_subdistricts.xlsx'")
    display(unresolved_out)
else:
    print("All cases resolved! No unresolved subDistricts.")
    # Create empty file for consistency
    pd.DataFrame(columns=unresolved_columns).to_excel(
        'unresolved_subdistricts.xlsx', index=False
    )

# Also export resolved cases for reference
resolved = results_df[results_df['inferred_subDistrict'].notna()].copy()

if not resolved.empty:
    resolved_out = resolved.sort_values(
        ['neutral_citation', 'charge_no']
    ).reset_index(drop=True).map(_sanitize_for_excel)
    resolved_out.to_excel('resolved_subdistricts.xlsx', index=False)
    print(f"Exported {len(resolved_out)} resolved cases to 'resolved_subdistricts.xlsx'")


# Also export full results for reference
results_df_sanitized = results_df.map(_sanitize_for_excel)
results_df_sanitized.to_excel('subdistrict_inference_results.xlsx', index=False)

print("Exported full results to 'subdistrict_inference_results.xlsx'")


Exported 126 unresolved cases to 'unresolved_subdistricts.xlsx'


,neutral_citation,charge_no,address,reason
0,[2021] HKCFI 1411,2,輕型貨車停於九龍塘達之路；及後於沙田一間酒店房間進行搜查並發現毒品,"The address mentions two different locations: 達之路 in 九龍塘 (which points to Kowloon Tong) and later a hotel room in 沙田 (which points to Sha Tin). Because the event spans multiple places and does not specify which single location should be assigned, a unique sub-district cannot be determined."
1,[2021] HKCFI 1919,2,the defendant's flat (residence),"The address 'the defendant's flat (residence)' provides no geographic clue or Hong Kong location information, so it cannot be matched to any listed sub-district."
2,[2021] HKCFI 2021,1,Post office,"""Post office"" is too vague and does not indicate any specific location in Hong Kong, so it cannot be mapped to a single known sub-district."
3,[2021] HKCFI 2396,2,Not given,"No address was provided ('Not given'), so the sub-district cannot be determined."
4,[2021] HKCFI 2596,3,garage,"The address 'garage' is too vague and provides no location clue in Hong Kong, so it cannot be mapped to any listed sub-district."
...,...,...,...,...
121,[2025] HKDC 615,1,香港羅湖管制站海關離境大堂,「羅湖管制站海關離境大堂」指的是羅湖口岸／Lo Wu Control Point。『羅湖』不在提供的有效子地區清單中；雖然其位置接近 Sheung Shui 一帶，但口岸本身無法明確對應到清單中的某一子地區，因此不應勉強推斷。
122,[2025] HKDC 615,2,羅湖管制站海關離境大堂,「羅湖管制站海關離境大堂」指羅湖管制站（Lo Wu Control Point）。雖然羅湖位於新界北部、鄰近上水／文錦渡一帶，但「羅湖」本身不在提供的有效子分區清單內，且不能明確對應到清單中的某一個子分區，因此不應勉強猜測。
123,[2025] HKDC 754,1,被告人的家中的睡房電腦枱上的塑膠抽屜,"The provided text only says 'the plastic drawer on the bedroom computer desk in the defendant’s home' and gives no Hong Kong location clue such as district, estate, street, building, or landmark, so the sub-district cannot be determined."
124,[2025] HKDC 954,1,被告於控罪所指的住所,"The address provided,「被告於控罪所指的住所」, is not an actual Hong Kong location but a generic legal phrase meaning 'the defendant's residence referred to in the charge'. It contains no identifiable geographic clue to map to any listed sub-district."


Exported 291 resolved cases to 'resolved_subdistricts.xlsx'
Exported full results to 'subdistrict_inference_results.xlsx'


In [16]:
# --- Export cases without address and subDistrict ---

# These are charges where both address and subDistrict are null/empty:
# the place of offence was simply not mentioned in the judgment.
# They differ from 'unresolved' cases (which have an address but GPT couldn't infer subDistrict).

pipeline_no_address = []
if active_only:
    pipeline_no_address.append({'$match': {'exclude': {'$ne': True}}})

pipeline_no_address.extend([
    {'$unwind': '$judgement.charges'},
    # Filter to charges where BOTH address and subDistrict are null/empty
    {'$match': {
        'judgement.charges.place_of_offence': {'$ne': None},
        '$expr': {
            '$and': [
                # address is null or empty
                {'$or': [
                    {'$eq': ['$judgement.charges.place_of_offence.address', None]},
                    {'$eq': ['$judgement.charges.place_of_offence.address', '']},
                ]},
                # subDistrict is null or empty
                {'$or': [
                    {'$eq': ['$judgement.charges.place_of_offence.subDistrict', None]},
                    {'$eq': ['$judgement.charges.place_of_offence.subDistrict', '']},
                ]},
            ]
        },
    }},
    {
        '$project': {
            '_id': 1,
            'neutral_citation': '$judgement.neutral_citation',
            'charge_no': '$judgement.charges.charge_no',
            'address': '$judgement.charges.place_of_offence.address',
            'nature': '$judgement.charges.place_of_offence.nature',
            'subDistrict': '$judgement.charges.place_of_offence.subDistrict',
        }
    },
])

rows_no_address = list(verified_features.aggregate(pipeline_no_address, allowDiskUse=True))
print(f"Found {len(rows_no_address)} charges with neither address nor subDistrict")

if rows_no_address:
    df_no_address = pd.DataFrame(rows_no_address)
    df_no_address['_id'] = df_no_address['_id'].apply(str)

    # Subtract charges already in the processed results
    processed_ids = set(zip(results_df['_id'], results_df['charge_no']))
    df_no_address = df_no_address[
        ~df_no_address.apply(lambda r: (r['_id'], r['charge_no']) in processed_ids, axis=1)
    ]

    columns = ['neutral_citation', 'charge_no', 'nature']

    if not df_no_address.empty:
        out = df_no_address[columns].sort_values(
            ['neutral_citation', 'charge_no']
        ).reset_index(drop=True)
        out = out.map(_sanitize_for_excel)
        out.to_excel('cases_without_address_subdistrict.xlsx', index=False)
        n = len(out)
        print(f"Exported {n} cases without address/subDistrict to cases_without_address_subdistrict.xlsx")
        display(out)
    else:
        print("All address-less cases were already captured in the processed results.")
else:
    print("No cases without address found.")


Found 134 charges with neither address nor subDistrict
Exported 134 cases without address/subDistrict to cases_without_address_subdistrict.xlsx


,neutral_citation,charge_no,nature
0,[2021] HKCFI 1393,1,Private vehicle
1,[2021] HKCFI 1963,1,Border checkpoint
2,[2021] HKCFI 2074,1,Private vehicle
3,[2021] HKCFI 2091,1,Residential building
4,[2021] HKCFI 2596,2,Private vehicle
...,...,...,...
129,[2025] HKDC 57,2,Private vehicle
130,[2025] HKDC 827,1,Residential building
131,[2025] HKDC 880,1,Residential building
132,[2025] HKDC 964,1,Public transport


In [15]:
# --- Summary ---

total = len(results_df)
resolved = results_df['inferred_subDistrict'].notna().sum()
unresolved_count = results_df['inferred_subDistrict'].isna().sum()

print("=" * 60)
print("SUMMARY: Infer Missing subDistricts")
print("=" * 60)
print(f"Total charges processed:  {total}")
print(f"Successfully inferred:    {resolved} ({resolved / total * 100:.1f}%)" if total else "N/A")
print(f"Still unresolved:         {unresolved_count} ({unresolved_count / total * 100:.1f}%)" if total else "N/A")
print()
if update_result:
    print(f"DB writes:               {update_result.modified_count} modified")
print()
if unresolved_count > 0:
    print(f"See 'unresolved_subdistricts.xlsx' for cases needing manual review.")
print(f"See 'subdistrict_inference_results.xlsx' for full results.")
print("=" * 60)


SUMMARY: Infer Missing subDistricts
Total charges processed:  417
Successfully inferred:    291 (69.8%)
Still unresolved:         126 (30.2%)

DB writes:               291 modified

See 'unresolved_subdistricts.xlsx' for cases needing manual review.
See 'subdistrict_inference_results.xlsx' for full results.
